<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_Solar_Accelerated_Discovery_of_Efficient_Solar_Cell_Materials_Using_Quantum_and_Machine_Learning_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Reference**: [Accelerated Discovery of Efficient Solar Cell Materials Using Quantum and Machine-Learning Methods](https://pubs.acs.org/doi/full/10.1021/acs.chemmater.9b02166)

In [ ]:
!pip install jarvis-tools

In [ ]:
from jarvis.db.figshare import data
import pandas as pd
df=pd.DataFrame(data('dft_3d'))

In [ ]:
df2=df[df['slme']!='na']

In [ ]:
df2.columns

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pylab as pylab
import matplotlib.pyplot as plt

def arr2pctarr(arr=[]):
  summ=np.sum(np.array(arr))
  new_arr=[100*float(i)/float(summ) for i in arr]
  return new_arr
  
sz=20
params = {'legend.fontsize': sz,
         'axes.labelsize': sz,
         'axes.titlesize':sz,
         'xtick.labelsize':sz,
         'ytick.labelsize':sz}
pylab.rcParams.update(params)
cutoff=34
intvl=1
hist_3d, bins_3d = np.histogram(df2['slme'],bins=np.arange(0, cutoff + intvl, intvl), density=False)
plt.rcParams['font.size'] = 20.0
y_tmp=arr2pctarr(hist_3d)
plt.bar(bins_3d[:-1],y_tmp,width=1)
plt.xlabel('$\eta$(%)')
plt.ylabel('%Materials')
plt.axvline(x=10.0)

plt.locator_params(axis='x', nbins=5)
plt.locator_params(axis='y', nbins=5)
#ax.annotate('', xy=(25,15),xytext=(15,15),arrowprops=dict(arrowstyle="->"))
plt.arrow(10,25,10,0,fc='red', ec='red',shape='full', length_includes_head=True,width = 0.015, edgecolor = 'red',  lw = 3,head_width=.5, head_length=0.5)           
  

In [ ]:
# df3=df2[['mbj_bandgap','slme']].replace('na',np.nan).dropna()
# plt.plot(df3['mbj_bandgap'],df3['slme'],'.')
# plt.xlim([.3,2.7])
# plt.ylim([10,35])

In [ ]:
df3=df2[['slme','effective_masses_300K']]
slme=[]
elec_eff_mass=[]
for i,j in df3.iterrows():
  if j['effective_masses_300K'] and 'n' in j['effective_masses_300K'] and j['effective_masses_300K']['n']!='na' and j['slme']>=10 :
    m=np.mean(np.array(j['effective_masses_300K']['n']))
    elec_eff_mass.append(m)
    slme.append(j['slme'])

In [ ]:
plt.scatter(elec_eff_mass,slme,s=5)
plt.xlim([0,3])
plt.ylim([10,35])
plt.axvline(x=1.0)
plt.xlabel('Electron effective mass')
plt.ylabel('$\eta$(%)')
plt.arrow(1,25,-0.75,0,fc='red', ec='red',shape='full', length_includes_head=True,width = 0.015, edgecolor = 'red',  lw = 3,head_width=0.5, head_length=0.05)


In [ ]:
from matplotlib.gridspec import GridSpec
import requests,zipfile,os,io,tempfile
from jarvis.io.vasp.outputs import Vasprun
from jarvis.analysis.solarefficiency.solar import SolarEfficiency


fls = data("raw_files")

def saturation_point(arr, tol=1):
    # assuming monotonic array
    maxa = max(arr)
    mina = min(arr)
    for i, ii in enumerate(arr):
        if abs(maxa - arr[i]) <= tol:
            return i


def get_tbmbj_vasprun(jid="JVASP-29640"):
    for i in fls["TBMBJ"]:
        if i["name"].split(".zip")[0] == jid:
            return i


jid='JVASP-29640'

plt.rcParams.update({"font.size": 20})
plt.figure(figsize=(5,4))

x = get_tbmbj_vasprun(jid=jid)
zip_file_url=x["download_url"]
print('zip_file_url',zip_file_url)
r = requests.get(zip_file_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
wdat = z.read("vasprun.xml").decode("utf-8")
fd, path = tempfile.mkstemp()
with os.fdopen(fd, "w") as tmp:
    tmp.write(wdat)
    v=Vasprun(path)

dirgap = v.get_dir_gap
indirgap = v.get_indir_gap[0]
en, abz = v.avg_absorption_coefficient
abz = np.array(abz) * 100
thickness = np.arange(1e-8, 5e-6, 1e-8)
effs = []
for i in thickness:

    eff = SolarEfficiency().slme(
        en, abz, dirgap, indirgap, plot_current_voltage=False, thickness=i,
        #en, abz, dirgap, indirgap, plot_current_voltage=False, thickness=i,
    )
    # eff = info["efficiency"]
    effs.append(eff*100)
optm = saturation_point(effs)
opt_thickness = thickness[optm]
opt_eff = effs[optm]

img = io.BytesIO()

plt.xlabel("Thickness(m)")
plt.ylabel("SLME(%)")
plt.plot(thickness, effs, ".")
plt.scatter([opt_thickness], [opt_eff], s=100, color="g")

# plt.subplot(the_grid[1])
# plt.title("(b)")
# temperatures=np.arange(10,100,5)
# effs=[]
# for t in temperatures:
#     info = tmp_slme(en, abz, indirgap, indirgap, plot_current_voltage=False,thickness=opt_thickness,temperature=t)
#     eff=info['efficiency']
#     effs.append(eff)
# optm=saturation_point(effs)
# opt_temp=temperatures[optm]
# opt_eff=effs[optm]
# plt.plot(temperatures,effs,'.')
# plt.scatter([opt_thickness],[opt_eff],s=100,color='g')
# info = SolarEfficiency().slme(
#     en,
#     abz,
#     dirgap,
#     indirgap,
#     plot_current_voltage=True,
#     thickness=opt_thickness,
# )
# plt.plot(info["V"], info["JV"])
# plt.ylabel("J (arb. unit)")
# plt.xlabel("Voltage(V)")
# plt.plot(info['V'],info['PV'],linestyle="--",label='PV')
# plt.legend()

# plt.tight_layout()
plt.show()
# plt.savefig(img, format="png")
# img.seek(0)
# plot_url = base64.b64encode(img.getvalue()).decode()
# print("opt_thickness,opt_eff", opt_thickness, opt_eff)

# return plot_url


In [ ]:
pip freeze